<a href="https://colab.research.google.com/github/EgueRichnel2025/gitmind-commit-model/blob/main/notebooks/02_chunking_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GitMind Commit Model — Chunking Experiment

Validation expérimentale du découpage de grands diffs avec le véritable
tokenizer Salesforce CodeT5-small.

Scénarios : 512, 5 000, 10 000 et 20 000 tokens, puis un commit
multifichier de 20 000 tokens.

Aucun GPU n'est nécessaire : cette expérience mesure uniquement
le traitement et la tokenisation.


In [2]:
%pip -q install \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "sentencepiece>=0.1.91,!=0.1.92"

## Redémarrage du runtime

Après l'installation, redémarrer la session Colab puis exécuter
les cellules suivantes dans l'ordre.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/gitmind-commit-model")

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/EgueRichnel2025/gitmind-commit-model.git",
            str(REPO_DIR),
        ],
        check=True,
    )

%cd /content/gitmind-commit-model

sys.path.insert(0, str(REPO_DIR / "src"))

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/gitmind-commit-model")
SOURCE_DIR = REPO_DIR / "src"

if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    part for part in (str(SOURCE_DIR), env.get("PYTHONPATH")) if part
)

command = [
    sys.executable,
    "scripts/experiment_chunking.py",
    "--model",
    "Salesforce/codet5-small",
    "--output",
    "artifacts/chunking_experiment.json",
]

print("===== CHUNKING EXPERIMENT =====")

result = subprocess.run(
    command,
    cwd=str(REPO_DIR),
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout or "(no standard output)")

if result.stderr:
    print("===== STDERR =====")
    print(result.stderr)

if result.returncode != 0:
    raise subprocess.CalledProcessError(
        result.returncode,
        command,
        output=result.stdout,
        stderr=result.stderr,
    )

print("Experiment completed successfully.")

In [ ]:
import json
from pathlib import Path

from google.colab import files

report_path = Path("artifacts/chunking_experiment.json")
report = json.loads(report_path.read_text(encoding="utf-8"))

print(json.dumps(report, indent=2, ensure_ascii=False))

files.download(str(report_path))